# SMT solvers — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def truth_rows(names):
    return [dict(zip(names, vals)) for vals in itertools.product([False, True], repeat=len(names))]
def draw_graph(nodes, edges, title='graph'):
    angles=np.linspace(0, 2*np.pi, len(nodes), endpoint=False)
    pos={n:(np.cos(a), np.sin(a)) for n,a in zip(nodes, angles)}
    plt.figure(figsize=(5,4))
    for a,b in edges:
        xa,ya=pos[a]; xb,yb=pos[b]; plt.plot([xa,xb],[ya,yb], 'k-', alpha=.6)
    for n,(x,y) in pos.items():
        plt.scatter([x],[y], s=500, zorder=3); plt.text(x,y,n,ha='center',va='center',color='white',weight='bold')
    plt.axis('equal'); plt.axis('off'); plt.title(title)
    return pos
print('setup ok')

## SAT search plus theory checks
SMT adds theories such as arithmetic to Boolean search. These examples show Boolean abstraction, linear arithmetic feasibility, conflicting bounds, theory propagation, and a branch-and-check loop.

In [ ]:
# 1) Boolean skeleton: P and Q abstracts two arithmetic atoms
rows=truth_rows(['P','Q']); sat=[r['P'] and r['Q'] for r in rows]
plt.figure(figsize=(5,3)); plt.bar(range(4), sat, color='tab:green'); plt.xticks(range(4), [f"{int(r['P'])}{int(r['Q'])}" for r in rows]); plt.title('Boolean abstraction P and Q')
assert sum(sat)==1

In [ ]:
# 2) theory check: x>=2 and x<=5 is feasible
xs=np.linspace(0,7,200); ok=(xs>=2)&(xs<=5)
plt.figure(figsize=(6,3)); plt.plot(xs,ok.astype(int)); plt.ylim(-.1,1.1); plt.title('feasible interval [2,5]')
assert ok.any() and abs(xs[ok][0]-2)<0.05

In [ ]:
# 3) conflicting bounds: x>=4 and x<=1 has no point
xs=np.linspace(0,5,100); ok=(xs>=4)&(xs<=1)
plt.figure(figsize=(6,3)); plt.plot(xs,ok.astype(int), color='tab:red'); plt.ylim(-.1,1.1); plt.title('empty theory region')
assert not ok.any()

In [ ]:
# 4) theory propagation: x>=2 and x<=5 implies x<=6
xs=np.linspace(0,7,200); base=(xs>=2)&(xs<=5); implied=(xs<=6)
plt.figure(figsize=(6,3)); plt.plot(xs,base.astype(int),label='base'); plt.plot(xs,implied.astype(int),label='x<=6',alpha=.7); plt.legend(); plt.title('all base points satisfy implied atom')
assert np.all(implied[base])

In [ ]:
# 5) branch and theory-check two Boolean candidates
cands=[{'P':1,'Q':1},{'P':1,'Q':0}]; feasible=[True,False]
plt.figure(figsize=(5,3)); plt.bar(['P,Q','P,not Q'], feasible, color=['tab:green','tab:red']); plt.title('SAT branch accepted only if theory-feasible')
assert feasible==[True, False]